[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/07_Parsing_and_Checker/Parsing_and_Checker_Apply.ipynb)

# 1.7 Parsing and Checker — Hands-On Practice

Practice using the ONNX parser, checker, and shape inference tools.

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | [Setup](#section-1) | Imports |
| 2 | [Exercise 1: Parse & Run a Model](#section-2) | Text format end-to-end |
| 3 | [Exercise 2: Break and Fix Models](#section-3) | Checker error diagnosis |
| 4 | [Exercise 3: Shape Inference](#section-4) | Propagate and inspect shapes |
| 5 | [Exercise 4: Parse Complex Model](#section-5) | Multi-layer with text format |
| 6 | [Exercise 5: Validation Pipeline](#section-6) | Reusable validate-and-report function |
| 7 | [Visualization: Shape Flow](#section-7) | Shape propagation diagram |
| 8 | [Challenge: Text Format MLP](#section-8) | Full MLP in text format |

<a id='section-1'></a>
## Section 1: Setup

In [ ]:
# !pip install onnx onnxruntime matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt

import onnx
import onnx.parser
from onnx import shape_inference, TensorProto
from onnx.checker import check_model
import onnxruntime as ort

print(f'ONNX: {onnx.__version__}, ORT: {ort.__version__}')

<a id='section-2'></a>
## Section 2: Exercise 1 — Parse & Run a Model

### Task

Define a ReLU model using the text format, parse it, validate it, and run inference.

In [ ]:
text = '''
    <ir_version: 8, opset_import: [ "" : 15 ]>
    relu_model (float[N,4] X) => (float[N,4] Y) {
        Y = Relu(X)
    }
'''

model = onnx.parser.parse_model(text)
check_model(model)

sess = ort.InferenceSession(
    model.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.array([[-1, 2, -3, 4], [5, -6, 7, -8]], dtype=np.float32)
result = sess.run(None, {'X': x})[0]

print('Parsed text format model:')
print(f'  Input:  {x}')
print(f'  Output: {result}')
print(f'  Match:  {np.allclose(result, np.maximum(0, x))}')

<a id='section-3'></a>
## Section 3: Exercise 2 — Break and Fix Models

### Task

Parse models with deliberate errors, observe the checker messages, and fix them.

In [ ]:
# Case 1: This model should work fine
ok_text = '''
    <ir_version: 8, opset_import: [ "" : 15 ]>
    g (float[N,4] X, float[4,2] W) => (float[N,2] Y) {
        Y = MatMul(X, W)
    }
'''

try:
    m = onnx.parser.parse_model(ok_text)
    check_model(m)
    print('Case 1 (valid MatMul): PASSED')
except Exception as e:
    print(f'Case 1: Error: {e}')

# Case 2: Wrong input count for Relu
try:
    bad_text = '''
        <ir_version: 8, opset_import: [ "" : 15 ]>
        g (float[N,4] X, float[N,4] Z) => (float[N,4] Y) {
            Y = Relu(X, Z)
        }
    '''
    m = onnx.parser.parse_model(bad_text)
    check_model(m)
    print('Case 2 (too many Relu inputs): PASSED (unexpected)')
except Exception as e:
    print(f'Case 2 (too many Relu inputs): {str(e)[:100]}')

In [ ]:
# Fixed version of Case 2
fixed_text = '''
    <ir_version: 8, opset_import: [ "" : 15 ]>
    g (float[N,4] X) => (float[N,4] Y) {
        Y = Relu(X)
    }
'''

m = onnx.parser.parse_model(fixed_text)
check_model(m)
print('Fixed Case 2: PASSED')

<a id='section-4'></a>
## Section 4: Exercise 3 — Shape Inference

### Task

Run shape inference on a multi-step model and report all intermediate shapes.

In [ ]:
text = '''
    <ir_version: 8, opset_import: [ "" : 15 ]>
    network (float[batch,4] X, float[4,8] W1, float[8,2] W2) => (float[batch,2] Y) {
        H = MatMul(X, W1)
        HR = Relu(H)
        Y = MatMul(HR, W2)
    }
'''

model = onnx.parser.parse_model(text)
inferred = shape_inference.infer_shapes(model)

def get_shape(type_proto):
    t = type_proto.tensor_type
    return [d.dim_param or d.dim_value for d in t.shape.dim]

print('Shape inference results:')
print(f'\n  Inputs:')
for inp in inferred.graph.input:
    print(f'    {inp.name:5s}: {get_shape(inp.type)}')

print(f'\n  Intermediates:')
for vi in inferred.graph.value_info:
    print(f'    {vi.name:5s}: {get_shape(vi.type)}')

print(f'\n  Outputs:')
for out in inferred.graph.output:
    print(f'    {out.name:5s}: {get_shape(out.type)}')

<a id='section-5'></a>
## Section 5: Exercise 4 — Parse Complex Model

### Task

Parse a model with multiple operations including Transpose using the text format.

In [ ]:
text = '''
    <ir_version: 8, opset_import: [ "" : 15 ]>
    transpose_lr (
        float[N,3] X,
        float[2,3] A,
        float[2] B
    ) => (float[N,2] Y) {
        tA = Transpose<perm = [1, 0]>(A)
        XA = MatMul(X, tA)
        Y = Add(XA, B)
    }
'''

model = onnx.parser.parse_model(text)
check_model(model)

# Run shape inference to see intermediate shapes
inferred = shape_inference.infer_shapes(model)

print('Transpose + MatMul model:')
for vi in inferred.graph.value_info:
    print(f'  {vi.name}: {get_shape(vi.type)}')

# Test it
sess = ort.InferenceSession(
    model.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.array([[1, 2, 3]], dtype=np.float32)
a = np.array([[0.5, -0.3, 0.1], [0.2, 0.8, -0.5]], dtype=np.float32)
b = np.array([0.1, -0.1], dtype=np.float32)

result = sess.run(None, {'X': x, 'A': a, 'B': b})[0]
expected = x @ a.T + b

print(f'\n  Result:   {result}')
print(f'  Expected: {expected}')
print(f'  Match: {np.allclose(result, expected)}')

<a id='section-6'></a>
## Section 6: Exercise 5 — Validation Pipeline

In [ ]:
def validate_model(text_or_model, run_shape_inference=True):
    """Full validation pipeline: parse, check, shape inference."""
    print('=' * 50)
    print('MODEL VALIDATION REPORT')
    print('=' * 50)

    # Step 1: Parse
    if isinstance(text_or_model, str):
        try:
            model = onnx.parser.parse_model(text_or_model)
            print('  [PASS] Parsing')
        except Exception as e:
            print(f'  [FAIL] Parsing: {e}')
            return None
    else:
        model = text_or_model
        print('  [SKIP] Parsing (already ModelProto)')

    # Step 2: Check
    try:
        check_model(model)
        print('  [PASS] Checker')
    except Exception as e:
        print(f'  [FAIL] Checker: {str(e)[:100]}')
        return model

    # Step 3: Shape inference
    if run_shape_inference:
        try:
            inferred = shape_inference.infer_shapes(model)
            n_shapes = len(inferred.graph.value_info)
            print(f'  [PASS] Shape inference ({n_shapes} intermediate shapes)')

            for vi in inferred.graph.value_info:
                shape = get_shape(vi.type)
                print(f'         {vi.name}: {shape}')
        except Exception as e:
            print(f'  [WARN] Shape inference: {str(e)[:100]}')

    # Summary
    print(f'\n  Graph: {model.graph.name}')
    print(f'  Nodes: {len(model.graph.node)}')
    print(f'  Inputs: {[i.name for i in model.graph.input]}')
    print(f'  Outputs: {[o.name for o in model.graph.output]}')
    print('=' * 50)
    return model

# Test with valid model
validate_model('''
    <ir_version: 8, opset_import: [ "" : 15 ]>
    test (float[N,4] X, float[4,2] W) => (float[N,2] Y) {
        Y = MatMul(X, W)
    }
''')

<a id='section-7'></a>
## Section 7: Visualization — Shape Flow

In [ ]:
text = '''
    <ir_version: 8, opset_import: [ "" : 15 ]>
    net (float[batch,784] X, float[784,256] W1, float[256,64] W2, float[64,10] W3)
        => (float[batch,10] Y) {
        H1 = MatMul(X, W1)
        H1r = Relu(H1)
        H2 = MatMul(H1r, W2)
        H2r = Relu(H2)
        Y = MatMul(H2r, W3)
    }
'''

model = onnx.parser.parse_model(text)
inferred = shape_inference.infer_shapes(model)

# Collect all shapes
all_shapes = {}
for inp in inferred.graph.input:
    all_shapes[inp.name] = get_shape(inp.type)
for vi in inferred.graph.value_info:
    all_shapes[vi.name] = get_shape(vi.type)
for out in inferred.graph.output:
    all_shapes[out.name] = get_shape(out.type)

# Data path tensors
path = ['X', 'H1', 'H1r', 'H2', 'H2r', 'Y']
widths = [all_shapes[n][-1] for n in path]

fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#3498DB', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6', '#1ABC9C']
for i, (name, w) in enumerate(zip(path, widths)):
    ax.barh(i, w, height=0.6, color=colors[i % len(colors)], alpha=0.7,
            edgecolor='white', linewidth=2)
    ax.text(w + 10, i, f'{name}: {all_shapes[name]}', va='center',
            fontsize=10, fontweight='bold')

ax.set_yticks(range(len(path)))
ax.set_yticklabels(path, fontsize=10)
ax.set_xlabel('Feature Dimension', fontsize=12)
ax.set_title('Shape Propagation: 784 → 256 → 64 → 10',
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

<a id='section-8'></a>
## Section 8: Challenge — Full MLP in Text Format

### Task

Define a complete 3-layer MLP in text format, run shape inference, validate, and execute.

In [ ]:
mlp_text = '''
    <ir_version: 8, opset_import: [ "" : 15 ]>
    full_mlp (
        float[batch, 4] X,
        float[4, 8] W1, float[8] b1,
        float[8, 4] W2, float[4] b2,
        float[4, 2] W3, float[2] b3
    ) => (float[batch, 2] Y) {
        L1 = MatMul(X, W1)
        L1b = Add(L1, b1)
        L1r = Relu(L1b)
        L2 = MatMul(L1r, W2)
        L2b = Add(L2, b2)
        L2r = Relu(L2b)
        L3 = MatMul(L2r, W3)
        Y = Add(L3, b3)
    }
'''

model = validate_model(mlp_text)

# Run inference
sess = ort.InferenceSession(
    model.SerializeToString(), providers=['CPUExecutionProvider'])

np.random.seed(42)
x = np.random.randn(10, 4).astype(np.float32)
w1 = np.random.randn(4, 8).astype(np.float32) * 0.1
b1 = np.zeros(8, dtype=np.float32)
w2 = np.random.randn(8, 4).astype(np.float32) * 0.1
b2 = np.zeros(4, dtype=np.float32)
w3 = np.random.randn(4, 2).astype(np.float32) * 0.1
b3 = np.zeros(2, dtype=np.float32)

result = sess.run(None, {
    'X': x, 'W1': w1, 'b1': b1, 'W2': w2, 'b2': b2, 'W3': w3, 'b3': b3})[0]

# NumPy reference
h1 = np.maximum(0, x @ w1 + b1)
h2 = np.maximum(0, h1 @ w2 + b2)
y_np = h2 @ w3 + b3

print(f'\nInference test:')
print(f'  Output shape: {result.shape}')
print(f'  Match NumPy:  {np.allclose(result, y_np, atol=1e-6)}')
print(f'  First 3 predictions: {result[:3].tolist()}')

---

## Summary

| Exercise | Skill |
|----------|-------|
| 1 | Parse and run models from text format |
| 2 | Diagnose and fix checker errors |
| 3 | Run and interpret shape inference |
| 4 | Complex models with attributes in text |
| 5 | Build a validation pipeline |
| 6 | Visualize shape propagation |
| Challenge | Full MLP in text format |

**Next:** [Evaluation and Runtime](../08_Evaluation_and_Runtime/) — ReferenceEvaluator, custom ops, benchmarking.